In [2]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from pprint import pprint
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tabulate import tabulate
import asyncio
import nest_asyncio

nest_asyncio.apply()

import os
import random

config = {
    "dataset":{
        "dti":"../../Data/scope_onside_common_v3.parquet",
        "adr":"../../Data/final_rxnorm_meddra_v2.parquet"
    },
    "protein_emb_1":{
        "path":  "../../Data/3. Protein_enbeddings/ESM_embeddings_(t33_650m model).parquet",
        "id_col": "id", 
        "emb_col": "embedding"
    },
    "protein_emb_2":{
        "path": "../../Data/3. Protein_enbeddings/GVP-GNN_protein_embeddings.parquet",
        "id_col": "uniprot_id", 
        "emb_col": "embedding"
    },
    "drug_emb_1":{
        "path": "../../Data/2. Drug_embeddings/EGNN_drug_embeddings_v2.parquet", 
        "id_col": "drug_chembl_id", 
        "emb_col": "embedding"
    },
    "drug_emb_2":{
        "path": "../../Data/2. Drug_embeddings/smiles_embeddings_chemberta.parquet", 
        "id_col": "drug_chembl_id", 
        "emb_col": "embedding"
    }
}

In [3]:
dti_df = pd.read_parquet(config["dataset"]["dti"])
print(dti_df.info())

# copy selected columns to a new df
# drug_chembl_id as drug_id and target_uniprot_id as protein_id
dti_df = dti_df.rename(columns={"drug_chembl_id": "drug_id", "target_uniprot_id": "protein_id"})

df = dti_df.copy()
df = df[["drug_id", "protein_id", "label", "rxcui"]]


if config["protein_emb_1"]["path"]:
    protein_emb_1_df = pd.read_parquet(config["protein_emb_1"]["path"])
    protein_emb_1_df = protein_emb_1_df.rename(columns={config["protein_emb_1"]["id_col"]: "protein_id"})
    df = df.merge(protein_emb_1_df[["protein_id", config["protein_emb_1"]["emb_col"]]], on="protein_id", how="left")
    df = df.rename(columns={config["protein_emb_1"]["emb_col"]: "prot_emb_1"})

if config["protein_emb_2"]["path"]:
    protein_emb_2_df = pd.read_parquet(config["protein_emb_2"]["path"])
    protein_emb_2_df = protein_emb_2_df.rename(columns={config["protein_emb_2"]["id_col"]: "protein_id"})
    df = df.merge(protein_emb_2_df[["protein_id", config["protein_emb_2"]["emb_col"]]], on="protein_id", how="left")
    df = df.rename(columns={config["protein_emb_2"]["emb_col"]: "prot_emb_2"})

if config["drug_emb_1"]["path"]:
    drug_emb_1_df = pd.read_parquet(config["drug_emb_1"]["path"])
    drug_emb_1_df = drug_emb_1_df.rename(columns={config["drug_emb_1"]["id_col"]: "drug_id"})
    df = df.merge(drug_emb_1_df[["drug_id", config["drug_emb_1"]["emb_col"]]], on="drug_id", how="left")
    df = df.rename(columns={config["drug_emb_1"]["emb_col"]: "drug_emb_1"})

if config["drug_emb_2"]["path"]:
    drug_emb_2_df = pd.read_parquet(config["drug_emb_2"]["path"])
    drug_emb_2_df = drug_emb_2_df.rename(columns={config["drug_emb_2"]["id_col"]: "drug_id"})
    df = df.merge(drug_emb_2_df[["drug_id", config["drug_emb_2"]["emb_col"]]], on="drug_id", how="left")
    df = df.rename(columns={config["drug_emb_2"]["emb_col"]: "drug_emb_2"})



print(df.info())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34741 entries, 0 to 34740
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   drug_chembl_id     34741 non-null  object
 1   target_uniprot_id  34741 non-null  object
 2   label              34741 non-null  int64 
 3   smiles             34741 non-null  object
 4   sequence           34741 non-null  object
 5   molfile_3d         34741 non-null  object
 6   rxcui              34741 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.9+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34741 entries, 0 to 34740
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   drug_id     34741 non-null  object
 1   protein_id  34741 non-null  object
 2   label       34741 non-null  int64 
 3   rxcui       34741 non-null  object
 4   prot_emb_1  34741 non-null  object
 5   prot_emb_2  34741 non

In [4]:

class ADRData:
    def __init__(self, id_to_name_dict):
        """
        Initializes with a dictionary of {meddra_id: meddra_name}.
        """
        self.id_to_name = id_to_name_dict
        self.unique_ids = sorted(list(id_to_name_dict.keys()))
        
        self.id_to_idx = {adr_id: i for i, adr_id in enumerate(self.unique_ids)}
        self.idx_to_id = {i: adr_id for i, adr_id in enumerate(self.unique_ids)}
        
        self.vocab_size = len(self.unique_ids)

    def encode(self, adr_list):
        """
        Takes a list of ADR IDs and returns a binary vector (1s and 0s).
        Example: ['10028553', '10003041'] -> [0, 1, 0, 0, 1...]
        """
        vector = np.zeros(self.vocab_size, dtype=np.int8)
        
        for adr_id in adr_list:
            if adr_id in self.id_to_idx:
                idx = self.id_to_idx[adr_id]
                vector[idx] = 1
            else:
                print(f"Warning: ADR ID {adr_id} not in vocabulary.")
                
        return vector

    def decode(self, vector):
        """
        Takes a binary vector and returns a list of human-readable ADR names.
        """
        decoded_names = []
        
        active_indices = np.where(vector == 1)[0]
        
        for idx in active_indices:
            adr_id = self.idx_to_id[idx]
            name = self.id_to_name.get(adr_id, "Unknown ADR")
            decoded_names.append(name)
            
        return decoded_names

    def decode_indices(self, indices):
        """
        Takes a list or array of indices (e.g., [42, 105, 300]) 
        and returns the corresponding ADR names.
        """
        return [self.id_to_name.get(self.idx_to_id[idx], "Unknown ADR") for idx in indices]

    def decode_top_k(self, confidence_array, k=5):
        """
        Takes the raw probability array from the model, finds the top K 
        highest values, and returns names + their confidence scores.
        """
        # Get indices of the top k probabilities
        top_indices = np.argsort(confidence_array)[-k:][::-1]
        
        results = []
        for idx in top_indices:
            adr_id = self.idx_to_id[idx]
            name = self.id_to_name.get(adr_id, "Unknown ADR")
            conf = confidence_array[idx]
            results.append({"name": name, "confidence": round(float(conf), 4)})
            
        return results

    def decode_with_threshold(self, confidence_array, threshold=0.5):
        """
        Returns all ADRs that pass a specific confidence threshold.
        Useful for seeing everything the model is "sure" about.
        """
        active_indices = np.where(confidence_array >= threshold)[0]
        
        # Sort them by confidence (highest first)
        active_indices = active_indices[np.argsort(confidence_array[active_indices])[::-1]]
        
        return [self.id_to_name.get(self.idx_to_id[idx], "Unknown ADR") for idx in active_indices]


In [5]:
adrdf = pd.read_parquet(config['dataset']["adr"])
id_name_dict = dict(zip(adrdf['meddra_id'], adrdf['meddra_name']))
adr_manager = ADRData(id_name_dict)

In [6]:
drug_to_adr_list = adrdf.groupby('rxnorm_ingredient_id')['meddra_id'].apply(list).to_dict()

def get_encoded_adr(drug_id):
    # Get the list of ADRs for this drug, or an empty list if not found
    adrs = drug_to_adr_list.get(drug_id, [])
    return adr_manager.encode(adrs)

# 2. Map the drug_id (e.g., rxcui) to the encoded vector
# This will create a column where each cell is a numpy array
df['adr'] = df['rxcui'].map(get_encoded_adr)
print(f"Total rows with ADRs: {df['adr'].apply(lambda x: x.sum() > 0).sum()}")

Total rows with ADRs: 34741


In [7]:
# print number of unique protein and drug ids

print(df["protein_id"].nunique())
print(df["drug_id"].nunique())

# print number of unique rxcui
print(df["rxcui"].nunique())
print(df.head(1))



2385
1028
1028
      drug_id protein_id  label  rxcui  \
0  CHEMBL1000     O15245      0  20610   

                                          prot_emb_1  \
0  [-0.041778564453125, 0.0305938720703125, -0.01...   

                                          prot_emb_2  \
0  [0.143310546875, 0.340087890625, -0.3349609375...   

                                          drug_emb_1  \
0  [0.02189382165670395, 0.016782937571406364, -0...   

                                          drug_emb_2  \
0  [0.008678080514073372, -0.08597195148468018, -...   

                                                 adr  
0  [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, ...  


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34741 entries, 0 to 34740
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   drug_id     34741 non-null  object
 1   protein_id  34741 non-null  object
 2   label       34741 non-null  int64 
 3   rxcui       34741 non-null  object
 4   prot_emb_1  34741 non-null  object
 5   prot_emb_2  34741 non-null  object
 6   drug_emb_1  34741 non-null  object
 7   drug_emb_2  34741 non-null  object
 8   adr         34741 non-null  object
dtypes: int64(1), object(8)
memory usage: 2.4+ MB


In [9]:
from sklearn.model_selection import train_test_split

# 1. Get all unique protein IDs
unique_proteins = df['protein_id'].unique()

# 2. Split protein IDs (not rows) to ensure no leakage
# We'll reserve 10% of proteins for Test and 10% for Validation
train_prot_ids, temp_prot_ids = train_test_split(
    unique_proteins, 
    test_size=0.20, 
    random_state=42
)

val_prot_ids, test_prot_ids = train_test_split(
    temp_prot_ids, 
    test_size=0.50, 
    random_state=42
)

# 3. Create the dataframes based on these ID splits
train_df = df[df['protein_id'].isin(train_prot_ids)]
val_df = df[df['protein_id'].isin(val_prot_ids)]
test_df = df[df['protein_id'].isin(test_prot_ids)]

# --- Verification & Metrics ---
print(f"--- Final Dataset Sizes ---")
print(f"Train Set: {len(train_df)} rows ({len(train_prot_ids)} proteins)")
print(f"Val Set:   {len(val_df)} rows ({len(val_prot_ids)} proteins) - [Model Selection]")
print(f"Test Set:  {len(test_df)} rows ({len(test_prot_ids)} proteins) - [Cold-Protein Eval]")

# 4. Recalculate Positive Weight for Training
num_neg = (train_df['label'] == 0).sum()
num_pos = (train_df['label'] == 1).sum()

# Avoid division by zero just in case
pos_weight_value = num_neg / num_pos if num_pos > 0 else 1.0

print(f"\nNew Positive Weight: {pos_weight_value:.2f}")
print(f"Positive/Negative Ratio in Train: 1:{num_neg/num_pos:.2f}")

--- Final Dataset Sizes ---
Train Set: 28055 rows (1908 proteins)
Val Set:   3335 rows (238 proteins) - [Model Selection]
Test Set:  3351 rows (239 proteins) - [Cold-Protein Eval]

New Positive Weight: 1.75
Positive/Negative Ratio in Train: 1:1.75


In [10]:
from sklearn.model_selection import train_test_split

# 1. Get all unique protein IDs
unique_drug = df['drug_id'].unique()

# 2. Split protein IDs (not rows) to ensure no leakage
# We'll reserve 10% of proteins for Test and 10% for Validation
train_drug_ids, temp_drug_ids = train_test_split(
    unique_drug, 
    test_size=0.20, 
    random_state=42
)

val_drug_ids, test_drug_ids = train_test_split(
    temp_drug_ids, 
    test_size=0.50, 
    random_state=42
)

# 3. Create the dataframes based on these ID splits
train_df = df[df['drug_id'].isin(train_drug_ids)]
val_df = df[df['drug_id'].isin(val_drug_ids)]
test_df = df[df['drug_id'].isin(test_drug_ids)]

# --- Verification & Metrics ---
print(f"--- Final Dataset Sizes ---")
print(f"Train Set: {len(train_df)} rows ({len(train_drug_ids)} drugs)")
print(f"Val Set:   {len(val_df)} rows ({len(val_drug_ids)} drugs) - [Model Selection]")
print(f"Test Set:  {len(test_df)} rows ({len(test_drug_ids)} drugs) - [Cold-Drug Eval]")

# 4. Recalculate Positive Weight for Training
num_neg = (train_df['label'] == 0).sum()
num_pos = (train_df['label'] == 1).sum()

# Avoid division by zero just in case
pos_weight_value = num_neg / num_pos if num_pos > 0 else 1.0

print(f"\nNew Positive Weight: {pos_weight_value:.2f}")
print(f"Positive/Negative Ratio in Train: 1:{num_neg/num_pos:.2f}")

--- Final Dataset Sizes ---
Train Set: 27976 rows (822 drugs)
Val Set:   3286 rows (103 drugs) - [Model Selection]
Test Set:  3479 rows (103 drugs) - [Cold-Drug Eval]

New Positive Weight: 1.81
Positive/Negative Ratio in Train: 1:1.81


In [16]:
# =========================
# ADR Baseline #1: OvR Logistic Regression (sklearn)
# =========================

import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score

# ---- 1) Feature builder: concatenate your embedding columns into one X ----
EMB_COLS = ["drug_emb_1", "drug_emb_2", "prot_emb_1", "prot_emb_2"]

def _to_1d_float(x):
    # each cell is expected to be a list/np.array/torch tensor
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    if hasattr(x, "detach"):  # torch tensor
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    return x.reshape(-1).astype(np.float32)

def build_X(df_in, emb_cols=EMB_COLS):
    mats = []
    for c in emb_cols:
        vecs = df_in[c].apply(_to_1d_float).tolist()
        if any(v is None for v in vecs):
            bad = sum(v is None for v in vecs)
            raise ValueError(
                f"[build_X] Column '{c}' has {bad} missing embeddings. "
                f"Fix embedding merges (left-join produced NaNs) before training."
            )
        mats.append(np.stack(vecs, axis=0))  # (N, dim_c)
    return np.concatenate(mats, axis=1)      # (N, sum_dims)

def build_Y(df_in):
    # df_in["adr"] already holds a multi-hot numpy vector per row
    return np.stack(df_in["adr"].values, axis=0).astype(np.int8)

X_train = build_X(train_df)
X_val   = build_X(val_df)
X_test  = build_X(test_df)

Y_train = build_Y(train_df)
Y_val   = build_Y(val_df)
Y_test  = build_Y(test_df)

print("[ADR] Shapes:",
      "X_train", X_train.shape, "Y_train", Y_train.shape,
      "| X_val", X_val.shape, "Y_val", Y_val.shape,
      "| X_test", X_test.shape, "Y_test", Y_test.shape)


# =========================
# ADR Baseline #0: Top-K Frequency Predictor
# =========================
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score

def safe_roc_auc(y_true, y_score, average="micro"):
    try:
        return roc_auc_score(y_true, y_score, average=average)
    except ValueError as e:
        print(f"[WARN] ROC-AUC undefined ({average}): {e}")
        return np.nan

def safe_pr_auc(y_true, y_score, average="micro"):
    try:
        return average_precision_score(y_true, y_score, average=average)
    except ValueError as e:
        print(f"[WARN] PR-AUC undefined ({average}): {e}")
        return np.nan

# ---- choose K (start small, then increase) ----
K = 200  # try 50/100/200/500

# ---- label frequencies on TRAIN ----
label_freq = Y_train.sum(axis=0)  # shape (L,)
topk_idx = np.argsort(-label_freq)[:K]  # top K labels by frequency

print(f"[TopK] Using K={K} | top label freq range:",
      int(label_freq[topk_idx[-1]]), "to", int(label_freq[topk_idx[0]]))

def topk_predict(Y_true, topk_idx, thr=0.5):
    """
    Return:
      - y_score: "prob-like" scores in [0,1] (1 for topK labels, 0 otherwise)
      - y_pred:  binary predictions (same as y_score>=thr)
    """
    n, L = Y_true.shape
    y_score = np.zeros((n, L), dtype=np.float32)
    y_score[:, topk_idx] = 1.0
    y_pred = (y_score >= thr).astype(np.int8)
    return y_score, y_pred

val_score, val_pred = topk_predict(Y_val, topk_idx)
test_score, test_pred = topk_predict(Y_test, topk_idx)

val_metrics = {
    "micro_aucroc": safe_roc_auc(Y_val, val_score, average="micro"),
    "micro_auprc":  safe_pr_auc(Y_val, val_score, average="micro"),
    "micro_f1":     f1_score(Y_val, val_pred, average="micro", zero_division=0),
    "macro_f1":     f1_score(Y_val, val_pred, average="macro", zero_division=0),
}

test_metrics = {
    "micro_aucroc": safe_roc_auc(Y_test, test_score, average="micro"),
    "micro_auprc":  safe_pr_auc(Y_test, test_score, average="micro"),
    "micro_f1":     f1_score(Y_test, test_pred, average="micro", zero_division=0),
    "macro_f1":     f1_score(Y_test, test_pred, average="macro", zero_division=0),
}

print("\n[ADR][TopK] Validation metrics:")
for k, v in val_metrics.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, (float, np.floating)) and not np.isnan(v) else f"  {k}: {v}")

print("\n[ADR][TopK] Test metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, (float, np.floating)) and not np.isnan(v) else f"  {k}: {v}")



[ADR] Shapes: X_train (27976, 2944) Y_train (27976, 4817) | X_val (3286, 2944) Y_val (3286, 4817) | X_test (3479, 2944) Y_test (3479, 4817)
[TopK] Using K=200 | top label freq range: 2642 to 25362

[ADR][TopK] Validation metrics:
  micro_aucroc: 0.787420
  micro_auprc: 0.158862
  micro_f1: 0.354854
  macro_f1: 0.015038

[ADR][TopK] Test metrics:
  micro_aucroc: 0.784321
  micro_auprc: 0.167178
  micro_f1: 0.368976
  macro_f1: 0.015466
